In [28]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os
import sys
sys.path.append('../../config')
from ceride_palettes import PALETTES

trials = pd.read_csv('../../data/norming_study/processed/b_pilot/trials.csv')

RESULTS_DIR = '../../results/norming_study/b_pilot'
os.makedirs(RESULTS_DIR, exist_ok=True)

In [29]:
# joint scatter: ability vs willingness w/ marginal histograms
item_means = (trials
    .groupby('itemID')[['abilityResponse', 'willingnessResponse']]
    .mean()
    .reset_index()
)

c0 = PALETTES['primary'][0]  # #789769
c1 = PALETTES['primary'][1]  # #596e6d
bg = '#e6dec7'
grid = '#d4ccb5'

import textwrap
def wrap(text, width=55):
    return '<br>'.join(textwrap.wrap(str(text), width))

has_action  = 'actionPhrase' in trials.columns
has_vignette = 'vignette' in trials.columns

if has_action:
    trials['_question_wrapped'] = trials['actionPhrase'].apply(lambda x: wrap(f'Can you {x}?'))
if has_vignette:
    trials['_vignette_wrapped'] = trials['vignette'].apply(wrap)

pad = 4  # padding so edge points aren't clipped

fig = make_subplots(
    rows=2, cols=2,
    column_widths=[0.8, 0.2],
    row_heights=[0.2, 0.8],
    shared_xaxes=True,
    shared_yaxes=True,
    horizontal_spacing=0.02,
    vertical_spacing=0.02,
)

# build hover
if has_vignette and has_action:
    customdata = np.stack([trials['_vignette_wrapped'], trials['_question_wrapped']], axis=1)
    hovertemplate = (
        '<i>%{customdata[0]}</i><br><br>'
        '<b>"%{customdata[1]}"</b><br><br>'
        'ability: %{x} &nbsp;|&nbsp; willingness: %{y}'
        '<extra></extra>'
    )
elif has_action:
    customdata = trials['_question_wrapped'].values.reshape(-1, 1)
    hovertemplate = (
        '<b>"%{customdata[0]}"</b><br><br>'
        'ability: %{x} &nbsp;|&nbsp; willingness: %{y}'
        '<extra></extra>'
    )
else:
    customdata = None
    hovertemplate = 'ability: %{x}<br>willingness: %{y}<extra></extra>'

scatter_kwargs = dict(
    x=trials['abilityResponse'],
    y=trials['willingnessResponse'],
    mode='markers',
    marker=dict(color=c0, size=5, opacity=0.5, line=dict(width=0)),
    hovertemplate=hovertemplate,
)
if customdata is not None:
    scatter_kwargs['customdata'] = customdata

fig.add_trace(go.Scatter(**scatter_kwargs), row=2, col=1)

# diagonal
fig.add_trace(go.Scatter(
    x=[0, 100], y=[0, 100],
    mode='lines',
    line=dict(color='#464548', width=0.8, dash='dash'),
    opacity=0.5,
    showlegend=False,
    hoverinfo='skip',
), row=2, col=1)

# top marginal (ability)
fig.add_trace(go.Histogram(
    x=trials['abilityResponse'],
    xbins=dict(start=0, end=100, size=5),
    marker_color=c0, opacity=0.85,
    showlegend=False,
), row=1, col=1)

# right marginal (willingness)
fig.add_trace(go.Histogram(
    y=trials['willingnessResponse'],
    ybins=dict(start=0, end=100, size=5),
    marker_color=c1, opacity=0.85,
    showlegend=False,
), row=2, col=2)

axis_style = dict(showgrid=True, gridcolor=grid, gridwidth=1,
                  linecolor='#464548', linewidth=1, ticks='outside',
                  tickcolor='#464548')

fig.update_layout(
    width=550, height=550,
    plot_bgcolor=bg,
    paper_bgcolor='white',
    showlegend=False,
    margin=dict(l=60, r=20, t=20, b=60),
    font=dict(family='Avenir, Helvetica Neue, Arial'),
    hoverlabel=dict(
        bgcolor='white',
        bordercolor='#464548',
        font=dict(family='Avenir, Helvetica Neue, Arial', size=13, color='#222'),
        namelength=0,
    ),
)
fig.update_xaxes(range=[-pad, 100+pad], title_text='ability', **axis_style, row=2, col=1)
fig.update_yaxes(range=[-pad, 100+pad], title_text='willingness', **axis_style, row=2, col=1)
fig.update_xaxes(showticklabels=False, showgrid=False, row=1, col=1)
fig.update_yaxes(showticklabels=False, showgrid=False, row=2, col=2)

fig.write_html(os.path.join(RESULTS_DIR, 'scatter_ability_willingness.html'))
fig.write_image(os.path.join(RESULTS_DIR, 'scatter_ability_willingness.png'), scale=2)
fig.show()